# HNeRV Stage 9 — Training on Colab A100

**Estimated time:** ~23 hours on A100 (Colab Pro)  
**Expected score:** 0.189–0.194 (likely beats leaderboard #1 at 0.192)

## Steps
1. Run Cell 1: Mount Drive + clone repo
2. Run Cell 2: Install deps
3. Run Cell 3: Start training (resumes automatically if session restarts)
4. Run Cell 4: Download archive.zip when done

**If Colab disconnects:** Just re-run Cell 3. It resumes from the last completed stage.

In [ ]:
# Cell 1: Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

import os

# ── EDIT THIS: your GitHub fork URL ──────────────────────────────────────────
REPO_URL = 'https://github.com/Bucky789/comma_video_compression_challenge.git'
# ─────────────────────────────────────────────────────────────────────────────

REPO_DIR = '/content/comma_video_compression_challenge'
DRIVE_CKPT = '/content/drive/MyDrive/hnerv_stage9_ckpts'

os.makedirs(DRIVE_CKPT, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    !cd {REPO_DIR} && git lfs pull
else:
    print('Repo already cloned, skipping.')

!nvidia-smi | head -12
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 2: Install dependencies
import subprocess, sys

!pip install -q av brotli timm einops segmentation-models-pytorch safetensors

# Check torch version
import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print('All deps installed OK')

In [ ]:
# Cell 3: Training with Drive checkpointing + auto-resume
#
# Checkpoints are saved to Drive after every stage.
# If this session disconnects, re-run this cell — it will resume from
# the last completed stage automatically.

import os, sys, time, shutil, json
from pathlib import Path
from datetime import datetime

REPO_DIR  = '/content/comma_video_compression_challenge'
DRIVE_CKPT = '/content/drive/MyDrive/hnerv_stage9_ckpts'

os.environ['COMMA_CHALLENGE_ROOT'] = REPO_DIR
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/submissions/hnerv_stage9/src')

import torch
from data import get_default_video_path
from stages.common import train_stage
from stages import (
    stage1_v328_ce, stage2_v331_softplus, stage3_v332_smooth, stage4_v332_qat,
    stage5_c1a_l7, stage6_lambda_sweep, stage7_sigma_sweep, stage8_muon_finetune,
    codec_stage,
)
from stages.stage9_latent_polish import run_latent_polish

device = torch.device('cuda')
video_path = get_default_video_path()
print(f'Video: {video_path}')
print(f'GPU:   {torch.cuda.get_device_name(0)}')

# Run dir: reuse existing Drive checkpoint dir if present
drive_run_marker = Path(DRIVE_CKPT) / 'run_name.txt'
if drive_run_marker.exists():
    run_name = drive_run_marker.read_text().strip()
    print(f'Resuming run: {run_name}')
else:
    run_name = f'run_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    drive_run_marker.write_text(run_name)
    print(f'New run: {run_name}')

out_root = Path(DRIVE_CKPT) / run_name
out_root.mkdir(parents=True, exist_ok=True)

def stage_done(path: Path) -> bool:
    return (path / 'final_decoder.pt').exists() and (path / 'final_latents.pt').exists()

shared_state = {}
t0 = time.time()

# ── Stages 1-8 ──────────────────────────────────────────────────────────────
builders = [
    stage1_v328_ce.make_config,
    stage2_v331_softplus.make_config,
    stage3_v332_smooth.make_config,
    stage4_v332_qat.make_config,
    stage5_c1a_l7.make_config,
    stage6_lambda_sweep.make_config,
    stage7_sigma_sweep.make_config,
    stage8_muon_finetune.make_config,  # 10,000 epochs
]
prev = None
for i, build in enumerate(builders, start=1):
    stage_out = out_root / f'stage{i}'
    if stage_done(stage_out):
        print(f'[Stage {i}] already done, skipping.')
        prev = stage_out
        continue
    cfg = build(stage_out) if i == 1 else build(prev, stage_out)
    result = train_stage(cfg, device, video_path=video_path, shared_state=shared_state)
    print(f'[Stage {i}] best={result["best_score"]:.4f} at ep{result["best_ep"]} '
          f'({(time.time()-t0)/3600:.1f}h elapsed)', flush=True)
    prev = stage_out

# ── Stage 9: latent polish ───────────────────────────────────────────────────
stage9_out = out_root / 'stage9'
if not stage_done(stage9_out):
    run_latent_polish(
        resume_from=prev, output_dir=stage9_out, device=device,
        video_path=video_path, epochs=2000, lr=1e-4, lr_floor=1e-7,
        batch_size=16, eval_every=25, ema_decay=0.999, shared_state=shared_state,
    )
    print(f'[Stage 9] done ({(time.time()-t0)/3600:.1f}h elapsed)', flush=True)
else:
    print('[Stage 9] already done, skipping.')

# ── Codec + frame selector ───────────────────────────────────────────────────
codec_out = out_root / 'submission_archive'
if not (codec_out / '0.bin').exists():
    r = codec_stage.run_codec_stage(stage9_out, codec_out, video_path,
                                     device=device, skip_selector=False)
    print(f'[codec] archive: {r["final_archive_bytes"]:,} bytes', flush=True)
else:
    print('[codec] already done, skipping.')

# Pack into archive.zip
import zipfile
zip_path = Path(DRIVE_CKPT) / 'archive.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(codec_out / '0.bin', '0.bin')
print(f'\nDone! archive.zip saved to Drive: {zip_path}')
print(f'Total wall-clock: {(time.time()-t0)/3600:.1f}h')

In [ ]:
# Cell 4: Download archive.zip to your computer
from google.colab import files
files.download('/content/drive/MyDrive/hnerv_stage9_ckpts/archive.zip')

In [ ]:
# Cell 5 (optional): Quick local eval to see score before submitting
import subprocess
result = subprocess.run(
    ['python', 'evaluate.py',
     '--submission-dir', f'{REPO_DIR}/submissions/hnerv_stage9',
     '--device', 'cuda'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout[-3000:])  # last 3000 chars
print(result.stderr[-1000:])